# Corpus Preprocessing (Defensive Approach)
**Goal:** Clean and normalize the raw corpus without destroying legal identifiers, structure, or benchmark integrity. We will output a new `documents_clean.parquet` file and leave the raw data untouched.

## Setup & Import Libraries

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# 1. Define Project Root and System Path
# Mặc định khi chạy notebook trong thư mục 'notebooks', thư mục gốc (root) sẽ nằm ở một cấp bên ngoài ('..')
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Thêm PROJECT_ROOT vào sys.path để Python có thể tìm thấy thư mục 'src'
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# 2. UI Settings
pd.set_option('display.max_colwidth', 800)

# 3. Import custom modules
from src.data.data_loader import load_parquet_data
from src.data.text_cleaner import normalize_vietnamese_text

## Section 1 — Load Raw Corpus
Load the `corpus.parquet` from the `raw/` directory. We will cast the `cid` column to a standard string type to prevent any `int`/`float` casting bugs later in the pipeline. We will also preserve the original text in a `text_raw` column to trace changes.

In [2]:
print("=== SECTION 1: LOAD RAW CORPUS ===")
corpus_path = f"{PROJECT_ROOT}/data/raw/YuITC/corpus.parquet"

try:
    corpus = load_parquet_data(corpus_path)
except Exception as e:
    print("Local load failed, falling back to HuggingFace URL...")
    corpus = pd.read_parquet("hf://datasets/YuITC/Vietnamese-Legal-Documents/corpus.parquet")

# Standardize CID to string to prevent datatype drift in vector databases/JSON later
corpus['cid'] = corpus['cid'].astype(str)

# Keep a pristine copy of the raw text for the final output schema and sanity checks
corpus.rename(columns={'text': 'text_raw'}, inplace=True)

print(f"Corpus shape: {corpus.shape}")
print("Data Types:")
print(corpus.dtypes)
display(corpus.head(2))

=== SECTION 1: LOAD RAW CORPUS ===
Corpus shape: (261597, 2)
Data Types:
text_raw    object
cid         object
dtype: object


,text_raw,cid
0,"Thông tư này hướng dẫn tuần tra, canh gác bảo vệ đê Điều trong mùa lũ đối với các tuyến đê sông được phân loại, phân cấp theo quy định tại Điều 4 của Luật Đê Điều.",0
1,"1. Hàng năm trước mùa mưa, lũ, Ủy ban nhân dân cấp xã nơi có đê phải tổ chức lực lượng lao động tại địa phương để tuần tra, canh gác đê và thường trực trên các điếm canh đê hoặc nhà dân khu vực gần đê (đối với những khu vực chưa có điếm canh đê), khi có báo động lũ từ cấp I trở lên đối với tuyến sông có đê (sau đây gọi tắt là lực lượng tuần tra, canh gác đê).\n2. Lực lượng tuần tra, canh gác đê được tổ chức thành các đội, do Ủy ban nhân dân cấp xã ra quyết định thành lập; từ 01 đến 02 kilômét đê thành lập 01 đội; mỗi đội có từ 12 đến 18 người, trong đó có 01 đội trưởng và 01 hoặc 02 đội phó. Danh sách thành viên đội tuần tra, canh gác đê được niêm yết tại điếm canh đê thuộc địa bàn được phân công.\n3. Khi lũ, bão có diễn biến phức tạp, kéo dài ngày, Uỷ ban nhân dân cấp xã có thể quyết ...",1


## Section 2 — Preprocessing Rules
We apply a strict "Defensive Preprocessing" philosophy:
1. **Unicode NFC:** Fixes character encoding inconsistencies (e.g., combined vs. precomposed vowels).
2. **Whitespace:** Collapses multiple spaces/tabs and normalizes newlines (`\r\n` -> `\n`) while preserving paragraph structures.
3. **Preservation:** No lowercasing, no stopword removal, no stemming, and NO removal of legal numbers/identifiers.

Let's test the cleaner on a messy, artificially corrupted dummy string.

In [3]:
print("=== SECTION 2: TEST PREPROCESSING RULES ===")

# Create a messy string with tabs, weird newlines, zero-width chars (\u200b), and inconsistent spacing
messy_dummy_text = "Điểm a \t Khoản 2  Điều   5\r\n\nNghị định  123/2020/NĐ-CP\u200b. \n\n\n  Theo   đó..."

print("[RAW TEXT]:")
print(repr(messy_dummy_text)) # Using repr() to show hidden characters like \t, \n

cleaned_dummy = normalize_vietnamese_text(messy_dummy_text)

print("\n[CLEANED TEXT]:")
print(repr(cleaned_dummy))

# Assertions to ensure our rules are working correctly
assert "Điểm a" in cleaned_dummy, "Failed: Destroyed single characters!"
assert "123/2020/NĐ-CP" in cleaned_dummy, "Failed: Destroyed legal numbers/identifiers!"
assert "\t" not in cleaned_dummy, "Failed: Did not remove tabs!"
assert "\u200b" not in cleaned_dummy, "Failed: Did not remove zero-width chars!"
print("\nText cleaner successfully passed defensive tests.")

=== SECTION 2: TEST PREPROCESSING RULES ===
[RAW TEXT]:
'Điểm a \t Khoản 2  Điều   5\r\n\nNghị định  123/2020/NĐ-CP\u200b. \n\n\n  Theo   đó...'

[CLEANED TEXT]:
'Điểm a Khoản 2 Điều 5\n\nNghị định 123/2020/NĐ-CP.\n\nTheo đó...'

Text cleaner successfully passed defensive tests.


## Section 3 — Unicode & Whitespace Normalization
Apply the `normalize_vietnamese_text` function across the entire dataset to create the new `text_clean` column. This may take a minute due to the scale of the corpus (261k+ rows).

In [4]:
print("=== SECTION 3: APPLY NORMALIZATION ===")

import time
start_time = time.time()

# Apply the cleaning function.
# We use .fillna("") to gracefully handle any stray NaN values without crashing.
corpus['text_clean'] = corpus['text_raw'].fillna("").apply(normalize_vietnamese_text)

elapsed = time.time() - start_time
print(f"Successfully cleaned {len(corpus):,} documents in {elapsed:.2f} seconds.")

# Display a brief comparison
display(corpus[['cid', 'text_raw', 'text_clean']].head(3))

=== SECTION 3: APPLY NORMALIZATION ===
Successfully cleaned 261,597 documents in 19.06 seconds.


,cid,text_raw,text_clean
0,0,"Thông tư này hướng dẫn tuần tra, canh gác bảo vệ đê Điều trong mùa lũ đối với các tuyến đê sông được phân loại, phân cấp theo quy định tại Điều 4 của Luật Đê Điều.","Thông tư này hướng dẫn tuần tra, canh gác bảo vệ đê Điều trong mùa lũ đối với các tuyến đê sông được phân loại, phân cấp theo quy định tại Điều 4 của Luật Đê Điều."
1,1,"1. Hàng năm trước mùa mưa, lũ, Ủy ban nhân dân cấp xã nơi có đê phải tổ chức lực lượng lao động tại địa phương để tuần tra, canh gác đê và thường trực trên các điếm canh đê hoặc nhà dân khu vực gần đê (đối với những khu vực chưa có điếm canh đê), khi có báo động lũ từ cấp I trở lên đối với tuyến sông có đê (sau đây gọi tắt là lực lượng tuần tra, canh gác đê).\n2. Lực lượng tuần tra, canh gác đê được tổ chức thành các đội, do Ủy ban nhân dân cấp xã ra quyết định thành lập; từ 01 đến 02 kilômét đê thành lập 01 đội; mỗi đội có từ 12 đến 18 người, trong đó có 01 đội trưởng và 01 hoặc 02 đội phó. Danh sách thành viên đội tuần tra, canh gác đê được niêm yết tại điếm canh đê thuộc địa bàn được phân công.\n3. Khi lũ, bão có diễn biến phức tạp, kéo dài ngày, Uỷ ban nhân dân cấp xã có thể quyết ...","1. Hàng năm trước mùa mưa, lũ, Ủy ban nhân dân cấp xã nơi có đê phải tổ chức lực lượng lao động tại địa phương để tuần tra, canh gác đê và thường trực trên các điếm canh đê hoặc nhà dân khu vực gần đê (đối với những khu vực chưa có điếm canh đê), khi có báo động lũ từ cấp I trở lên đối với tuyến sông có đê (sau đây gọi tắt là lực lượng tuần tra, canh gác đê).\n2. Lực lượng tuần tra, canh gác đê được tổ chức thành các đội, do Ủy ban nhân dân cấp xã ra quyết định thành lập; từ 01 đến 02 kilômét đê thành lập 01 đội; mỗi đội có từ 12 đến 18 người, trong đó có 01 đội trưởng và 01 hoặc 02 đội phó. Danh sách thành viên đội tuần tra, canh gác đê được niêm yết tại điếm canh đê thuộc địa bàn được phân công.\n3. Khi lũ, bão có diễn biến phức tạp, kéo dài ngày, Uỷ ban nhân dân cấp xã có thể quyết ..."
2,2,"Tiêu chuẩn của các thành viên thuộc lực lượng tuần tra, canh gác đê\n1. Là người khoẻ mạnh, tháo vát, đủ khả năng đảm đương những công việc nặng nhọc, kể cả lúc mưa to, gió lớn, đêm tối.\n2. Có tinh thần trách nhiệm, chịu đựng gian khổ, khắc phục khó khăn, quen sông nước và biết bơi, có kiến thức, kinh nghiệm hộ đê, phòng, chống lụt, bão.","Tiêu chuẩn của các thành viên thuộc lực lượng tuần tra, canh gác đê\n1. Là người khoẻ mạnh, tháo vát, đủ khả năng đảm đương những công việc nặng nhọc, kể cả lúc mưa to, gió lớn, đêm tối.\n2. Có tinh thần trách nhiệm, chịu đựng gian khổ, khắc phục khó khăn, quen sông nước và biết bơi, có kiến thức, kinh nghiệm hộ đê, phòng, chống lụt, bão."


## Section 4 — Artifact Cleaning
We will check if there are any residual HTML tags (like `<p>`, `<br>`, `&nbsp;`). If the dataset is already plain text, this step will safely skip. If it finds HTML, it will clean it gently without destroying mathematical operators (e.g., `a < b`).

In [5]:
print("=== SECTION 4: ARTIFACT CLEANING ===")
import re

# Safely search for common HTML tags and entities
html_mask = corpus['text_clean'].str.contains(r'<[a-z/][^>]*>|&[a-z]+;', regex=True, case=False)
html_count = html_mask.sum()

print(f"Documents containing potential HTML artifacts: {html_count:,}")

if html_count > 0:
    # 1. Replace <br> with newlines
    corpus['text_clean'] = corpus['text_clean'].str.replace(r'<br\s*/?>', '\n', regex=True, flags=re.IGNORECASE)
    # 2. Replace &nbsp; with space
    corpus['text_clean'] = corpus['text_clean'].str.replace(r'&nbsp;', ' ', regex=True, flags=re.IGNORECASE)
    # 3. Strip remaining HTML tags
    corpus['text_clean'] = corpus['text_clean'].str.replace(r'<[^>]+>', '', regex=True)
    # 4. Re-normalize to fix any spacing issues caused by tag removal
    corpus['text_clean'] = corpus['text_clean'].apply(normalize_vietnamese_text)
    print("HTML artifacts gently cleaned.")
else:
    print("No significant HTML artifacts found. Safe to proceed.")

=== SECTION 4: ARTIFACT CLEANING ===
Documents containing potential HTML artifacts: 155
HTML artifacts gently cleaned.


## Section 5 — Quality Flags
Instead of deleting records, we will append boolean flags to help us understand and handle edge cases during Retrieval and RAG Generation.
- `is_very_short`: < 5 words.
- `needs_local_passage_selection`: > 1000 words (Context window warning).
- `is_punctuation_only`: Contains no letters or numbers.

In [6]:
print("=== SECTION 5: QUALITY FLAGS ===")

# Calculate lengths on the cleaned text
corpus['char_count'] = corpus['text_clean'].str.len()
corpus['word_count'] = corpus['text_clean'].str.split().str.len()

# Flag 1: Very short documents
corpus['is_very_short'] = corpus['word_count'] < 5

# Flag 2: Very long documents (Requires local chunking later to fit LLM window)
corpus['needs_local_passage_selection'] = corpus['word_count'] > 1000

# Flag 3: Punctuation/Garbage only
def is_punct_only(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return True
    # Remove all non-alphanumeric characters. If length is 0, it's punctuation only.
    cleaned = re.sub(r'[\W_]+', '', text)
    return len(cleaned) == 0

corpus['is_punctuation_only'] = corpus['text_clean'].apply(is_punct_only)

print("Quality flags generated successfully:")
print(f"- is_very_short (< 5 words):                  {corpus['is_very_short'].sum():,}")
print(f"- needs_local_passage_selection (> 1k words): {corpus['needs_local_passage_selection'].sum():,}")
print(f"- is_punctuation_only:                        {corpus['is_punctuation_only'].sum():,}")

=== SECTION 5: QUALITY FLAGS ===
Quality flags generated successfully:
- is_very_short (< 5 words):                  34
- needs_local_passage_selection (> 1k words): 1,993
- is_punctuation_only:                        7


## Section 6 — Before/After Inspection
Let's visually inspect cases where the cleaner actually modified the text to ensure it only fixed formatting (spacing, invisible chars, newlines) without altering legal content.

In [7]:
print("=== SECTION 6: BEFORE/AFTER INSPECTION ===")

# Identify documents that were actually modified by the cleaner
changed_mask = corpus['text_raw'] != corpus['text_clean']
changed_count = changed_mask.sum()

print(f"Total documents modified during cleaning: {changed_count:,} / {len(corpus):,} ({(changed_count/len(corpus))*100:.1f}%)")

if changed_count > 0:
    print("\n--- Sample of modified documents (using repr() to show hidden chars) ---")
    # Sample 3 random modified documents
    sample_changed = corpus[changed_mask].sample(3, random_state=42)

    for _, row in sample_changed.iterrows():
        print(f"\n[CID: {row['cid']}]")
        print(f"RAW   : {repr(row['text_raw'][:150])}...")
        print(f"CLEAN : {repr(row['text_clean'][:150])}...")

print("\n--- Inspecting the final DataFrame structure ---")
display(corpus.head(3))

=== SECTION 6: BEFORE/AFTER INSPECTION ===
Total documents modified during cleaning: 8,173 / 261,597 (3.1%)

--- Sample of modified documents (using repr() to show hidden chars) ---

[CID: 65791]
RAW   : 'Bán, thanh lý, tiêu hủy tài sản công và xử lý tài sản công trong trường hợp bị mất, bị hủy hoại\n...\n2. Nội dung, trình tự, thủ tục bán, thanh lý, tiêu'...
CLEAN : 'Bán, thanh lý, tiêu hủy tài sản công và xử lý tài sản công trong trường hợp bị mất, bị hủy hoại\n...\n2. Nội dung, trình tự, thủ tục bán, thanh lý, tiêu'...

[CID: 94020]
RAW   : 'Đầu tư xây dựng trụ sở làm việc của cơ quan nhà nước\n...\n2. Trụ sở làm việc được đầu tư xây dựng theo một trong các mô hình sau đây:\na) Khu hành chính'...
CLEAN : 'Đầu tư xây dựng trụ sở làm việc của cơ quan nhà nước\n...\n2. Trụ sở làm việc được đầu tư xây dựng theo một trong các mô hình sau đây:\na) Khu hành chính'...

[CID: 104831]
RAW   : 'Giao quyền trong xử phạt vi phạm hành chính\n 1. Quyết định giao quyền quy định tại Điều 54, khoản 2

,text_raw,cid,text_clean,char_count,word_count,is_very_short,needs_local_passage_selection,is_punctuation_only
0,"Thông tư này hướng dẫn tuần tra, canh gác bảo vệ đê Điều trong mùa lũ đối với các tuyến đê sông được phân loại, phân cấp theo quy định tại Điều 4 của Luật Đê Điều.",0,"Thông tư này hướng dẫn tuần tra, canh gác bảo vệ đê Điều trong mùa lũ đối với các tuyến đê sông được phân loại, phân cấp theo quy định tại Điều 4 của Luật Đê Điều.",163,37,False,False,False
1,"1. Hàng năm trước mùa mưa, lũ, Ủy ban nhân dân cấp xã nơi có đê phải tổ chức lực lượng lao động tại địa phương để tuần tra, canh gác đê và thường trực trên các điếm canh đê hoặc nhà dân khu vực gần đê (đối với những khu vực chưa có điếm canh đê), khi có báo động lũ từ cấp I trở lên đối với tuyến sông có đê (sau đây gọi tắt là lực lượng tuần tra, canh gác đê).\n2. Lực lượng tuần tra, canh gác đê được tổ chức thành các đội, do Ủy ban nhân dân cấp xã ra quyết định thành lập; từ 01 đến 02 kilômét đê thành lập 01 đội; mỗi đội có từ 12 đến 18 người, trong đó có 01 đội trưởng và 01 hoặc 02 đội phó. Danh sách thành viên đội tuần tra, canh gác đê được niêm yết tại điếm canh đê thuộc địa bàn được phân công.\n3. Khi lũ, bão có diễn biến phức tạp, kéo dài ngày, Uỷ ban nhân dân cấp xã có thể quyết ...",1,"1. Hàng năm trước mùa mưa, lũ, Ủy ban nhân dân cấp xã nơi có đê phải tổ chức lực lượng lao động tại địa phương để tuần tra, canh gác đê và thường trực trên các điếm canh đê hoặc nhà dân khu vực gần đê (đối với những khu vực chưa có điếm canh đê), khi có báo động lũ từ cấp I trở lên đối với tuyến sông có đê (sau đây gọi tắt là lực lượng tuần tra, canh gác đê).\n2. Lực lượng tuần tra, canh gác đê được tổ chức thành các đội, do Ủy ban nhân dân cấp xã ra quyết định thành lập; từ 01 đến 02 kilômét đê thành lập 01 đội; mỗi đội có từ 12 đến 18 người, trong đó có 01 đội trưởng và 01 hoặc 02 đội phó. Danh sách thành viên đội tuần tra, canh gác đê được niêm yết tại điếm canh đê thuộc địa bàn được phân công.\n3. Khi lũ, bão có diễn biến phức tạp, kéo dài ngày, Uỷ ban nhân dân cấp xã có thể quyết ...",858,199,False,False,False
2,"Tiêu chuẩn của các thành viên thuộc lực lượng tuần tra, canh gác đê\n1. Là người khoẻ mạnh, tháo vát, đủ khả năng đảm đương những công việc nặng nhọc, kể cả lúc mưa to, gió lớn, đêm tối.\n2. Có tinh thần trách nhiệm, chịu đựng gian khổ, khắc phục khó khăn, quen sông nước và biết bơi, có kiến thức, kinh nghiệm hộ đê, phòng, chống lụt, bão.",2,"Tiêu chuẩn của các thành viên thuộc lực lượng tuần tra, canh gác đê\n1. Là người khoẻ mạnh, tháo vát, đủ khả năng đảm đương những công việc nặng nhọc, kể cả lúc mưa to, gió lớn, đêm tối.\n2. Có tinh thần trách nhiệm, chịu đựng gian khổ, khắc phục khó khăn, quen sông nước và biết bơi, có kiến thức, kinh nghiệm hộ đê, phòng, chống lụt, bão.",338,71,False,False,False
